In [3]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
# If you're running the notebook from a subfolder, walk up until we find "data"
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import requests
from bs4 import BeautifulSoup
response = requests.get(url, headers=headers)

In [15]:
from data.bme import fetch
import pandas as pd
import requests

url = "https://en.wikipedia.org/wiki/IBEX_35"
url2 = "https://es.wikipedia.org/wiki/IBEX_35"
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
tables = pd.read_html(response.text)
df = None
# tables = pd.read_html(url)
for table in tables:
    if 'Ticker' in table.columns and 'Company' in table.columns:
        df = table.copy()

df

/var/folders/bc/hnzwjdn546lcc572zg36k_vc0000gn/T/ipykernel_62915/4110181265.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


,Ticker,Company,Sector
0,ACS.MC,ACS,Construction
1,ACX.MC,Acerinox,Steel
2,AMS.MC,Amadeus IT Group,Tourism
3,ANA.MC,Acciona,Construction
4,ANE.MC,Acciona Energía,Energy
5,BBVA.MC,BBVA,Financial Services
6,BKT.MC,Bankinter,Financial Services
7,CABK.MC,CaixaBank,Financial Services
8,CLNX.MC,Cellnex Telecom,Telecommunications
9,COL.MC,Inmobiliaria Colonial,Real Estate


In [16]:
import yfinance as yf

def get_isin(ticker):
    try:
        info = yf.Ticker(ticker).info
        return info.get("isin", "")
    except:
        return ""

df["ISIN"] = df["Ticker"].apply(get_isin)
df

HTTP Error 500: {"quoteSummary":{"result":null,"error":{"code":"internal-error","description":"internal-error"}}}


,Ticker,Company,Sector,ISIN
0,ACS.MC,ACS,Construction,
1,ACX.MC,Acerinox,Steel,
2,AMS.MC,Amadeus IT Group,Tourism,
3,ANA.MC,Acciona,Construction,
4,ANE.MC,Acciona Energía,Energy,
5,BBVA.MC,BBVA,Financial Services,
6,BKT.MC,Bankinter,Financial Services,
7,CABK.MC,CaixaBank,Financial Services,
8,CLNX.MC,Cellnex Telecom,Telecommunications,
9,COL.MC,Inmobiliaria Colonial,Real Estate,


In [31]:
ALPHA_VANTAGE_KEY = "OES1IQGDSQCVOVQV"

import requests
import pandas as pd
from io import StringIO

# API_KEY = "YOUR_ALPHA_VANTAGE_KEY"

def fetch_daily_csv(symbol: str) -> pd.DataFrame | None:
    url = (
        "https://www.alphavantage.co/query"
        f"?function=TIME_SERIES_DAILY"
        f"&symbol={symbol}"
        f"&datatype=csv"
        f"&apikey={ALPHA_VANTAGE_KEY}"
    )
    r = requests.get(url, timeout=30)
    text = r.text.strip()

    # Alpha Vantage often returns an error message as plain text/JSON-like text
    if not text or "Error Message" in text or "Note" in text or "Information" in text:
        print(f"Failed for {symbol}: {text[:200]}")
        return None

    try:
        df = pd.read_csv(StringIO(text))
    except Exception as e:
        print(f"CSV parse failed for {symbol}: {e}")
        return None

    if "timestamp" not in df.columns or "close" not in df.columns:
        print(f"No daily data columns for {symbol}")
        return None

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["return"] = df["close"].pct_change()
    return df

# Try a few candidate symbol formats for one stock
candidates = ["SAB.BME", "SAB.ES"]

for sym in candidates:
    df = fetch_daily_csv(sym)
    if df is not None and not df.empty:
        print(f"Success with {sym}")
        print(df.tail())
        break

Failed for SAB.BME: {
    "Error Message": "Invalid API call. Please retry or visit the documentation (https://www.alphavantage.co/documentation/) for TIME_SERIES_DAILY."
}
Failed for SAB.ES: {
    "Information": "Thank you for using Alpha Vantage! Please consider spreading out your free API requests more sparingly (1 request per second). You may subscribe to any of the premium plans at ht


In [51]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
# url = f'https://www.alphavantage.co/query?function=SYMBOL_SEARCH&keywords=SAN&apikey={ALPHA_VANTAGE_KEY}'
url = 'https://eodhd.com/api/eod/SAB.MAD?api_token=demo&fmt=json'
r = requests.get(url)
data = r.json()
data[-1]
# for i in data['bestMatches']:
#     print(i['4. region'])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [24]:
from datetime import datetime
now = datetime.now()
date_time = now.strftime('%Y%m%d-%H:%M:%S')
date_time

'20260330-19:00:23'

In [ ]:
I can use ib script to do so

,AAPL,MSFT,AMZN,META
Date,,,,
2015-07-24,-0.005273,-0.003687,0.097972,0.015821
2015-07-27,-0.013896,-0.012843,0.003759,-0.028675
2015-07-28,0.004969,-0.000220,-0.010124,0.011893
2015-07-29,-0.003161,0.020953,0.005646,0.017840
2015-07-30,-0.005041,0.012746,0.014669,-0.018352
...,...,...,...,...
2017-02-16,-0.001181,-0.000155,0.001709,0.002998
2017-02-17,0.002734,0.001550,0.001102,-0.002316
2017-02-21,0.007221,-0.002012,0.013455,0.001423
